In [ ]:

# Explainable Supervised Anomaly Classification on 4 Industrial Datasets with DINOv2

# Authors: Hedieh Sajedi , Abolfazl Khojasteh Abkenar

!pip install -q transformers scikit-learn shap seaborn statsmodels tabulate kaggle scipy

import os, gc, time, types, json, warnings, tarfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              confusion_matrix, average_precision_score, matthews_corrcoef)
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import seaborn as sns
import shap
from scipy.ndimage import zoom as ndi_zoom, gaussian_filter as ndi_gaussian_filter, label as ndi_label

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)

ASSETS_DIR = "./paper_assets"
os.makedirs(ASSETS_DIR, exist_ok=True)
os.makedirs(f"{ASSETS_DIR}/figures", exist_ok=True)  # top-level: flowchart + cross-dataset summary



BASE_DATASETS_DIR = "./industrial_datasets"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
IMG_SIZE = 336
RESIZE_SIZE = 384
USE_FP16 = True

SUPERVISED_CV_FOLDS = 5
RANDOM_STATE = 42


XAI_SAMPLES_PER_CATEGORY = 3       
XAI_GALLERY_N_IMAGES = 3           
OCCLUSION_GRID = 7                  
DISCARD_RATIO_ROLLOUT = 0.0         
BOOTSTRAP_ITERS = 1000              


GOOD_NAMES = ("good", "ok", "normal")


DEBUG_MODE = False
DEBUG_N_CATEGORIES = 2
DEBUG_XAI_SAMPLES = 1
if DEBUG_MODE:
    XAI_SAMPLES_PER_CATEGORY = DEBUG_XAI_SAMPLES
    BOOTSTRAP_ITERS = 50
    print(f"DEBUG_MODE is ON - limiting to {DEBUG_N_CATEGORIES} categories/dataset, "
          f"{DEBUG_XAI_SAMPLES} XAI sample(s)/category. Set DEBUG_MODE=False for the real run.")

print(f"Device: {DEVICE}")


DATASETS = {
    "mvtec_ad": {
        "display_name": "MVTec AD",
        "kaggle_slug": "ipythonx/mvtec-ad",
        "root_subdir": "MVTec_AD",
    },
    "mvtec_loco_ad": {
        "display_name": "MVTec LOCO AD",
        "url": ("https://www.mydrive.ch/shares/48237/1b9106ccdfbb09a0c414bd49fe44a14a/"
                "download/430647091-1646842701/mvtec_loco_anomaly_detection.tar.xz"),
        "root_subdir": "MVTec_LOCO_AD",
    },
    "mpdd": {
        "display_name": "MPDD (Metal Parts Defect Detection)",
        "kaggle_slug": "lephonghao/metal-parts-defect-detection-dataset-mpdd",
        "root_subdir": "MPDD",
    },
    "btad": {
        "display_name": "BTAD (BeanTech Anomaly Detection)",
        "kaggle_slug": "thtuan/btad-beantech-anomaly-detection",
        "root_subdir": "BTAD",
    },
}



def ensure_kaggle_credentials():
    if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        return
    from google.colab import files
    print("Upload your kaggle.json:")
    files.upload()
    os.system("mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
    os.system("pip install -q --upgrade kaggle")

def find_category_dirs_generic(root):
    cats = []
    if not os.path.isdir(root):
        return cats
    for dirpath, dirnames, _ in os.walk(root):
        if "train" in dirnames and "test" in dirnames:
            train_dir = os.path.join(dirpath, "train")
            sub = [d.lower() for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
            if any(s in GOOD_NAMES for s in sub):
                cats.append(dirpath)
    return sorted(cats)

def get_mvtec_style_paths(cat_dir):
    
    train_root = os.path.join(cat_dir, "train")
    good_name = next((c for c in GOOD_NAMES if os.path.isdir(os.path.join(train_root, c))), None)
    train_paths = []
    if good_name:
        gd = os.path.join(train_root, good_name)
        train_paths = [os.path.join(gd, f) for f in sorted(os.listdir(gd)) if f.lower().endswith(IMG_EXTS)]

    test_root = os.path.join(cat_dir, "test")
    test_paths, test_labels, test_defect_types = [], [], []
    if os.path.isdir(test_root):
        for defect_type in sorted(os.listdir(test_root)):
            d = os.path.join(test_root, defect_type)
            if not os.path.isdir(d):
                continue
            label = 0 if defect_type.lower() in GOOD_NAMES else 1
            for f in sorted(os.listdir(d)):
                if f.lower().endswith(IMG_EXTS):
                    test_paths.append(os.path.join(d, f))
                    test_labels.append(label)
                    test_defect_types.append(defect_type)
    return train_paths, test_paths, test_labels, test_defect_types


def download_mvtec():
    root = os.path.join(BASE_DATASETS_DIR, DATASETS["mvtec_ad"]["root_subdir"])
    os.makedirs(BASE_DATASETS_DIR, exist_ok=True)
    if find_category_dirs_generic(root):
        print("  MVTec AD already present.")
        return root
    print("  MVTec AD not found locally. Downloading from Kaggle...")
    ensure_kaggle_credentials()
    os.system(f"kaggle datasets download -d {DATASETS['mvtec_ad']['kaggle_slug']} -p {root} --unzip")
    if not find_category_dirs_generic(root):
        raise RuntimeError(f"MVTec AD download failed or has an unexpected folder layout - "
                            f"inspect the extracted files under {root}.")
    return root

def download_mvtec_loco():
    root = os.path.join(BASE_DATASETS_DIR, DATASETS["mvtec_loco_ad"]["root_subdir"])
    os.makedirs(root, exist_ok=True)
    if find_category_dirs_generic(root):
        print("  MVTec LOCO AD already present.")
        return root
    print("  MVTec LOCO AD not found locally. Downloading (~3.6 GB) via direct link...")
    url = DATASETS["mvtec_loco_ad"]["url"]
    archive_path = os.path.join(root, "mvtec_loco_anomaly_detection.tar.xz")
    os.system(f"wget -q -O '{archive_path}' '{url}'")
    if not os.path.exists(archive_path) or os.path.getsize(archive_path) < 10**6:
        raise RuntimeError(
            "MVTec LOCO AD download failed (file missing or too small). The mydrive.ch link may "
            "require a browser session/cookie that wget can't replicate. Download the .tar.xz "
            "manually from https://www.mvtec.com/company/research/datasets/mvtec-loco , "
            f"place it at {archive_path}, then re-run this cell.")
    print("  Extracting...")
    with tarfile.open(archive_path) as tf:
        tf.extractall(root)
    if not find_category_dirs_generic(root):
        raise RuntimeError("MVTec LOCO AD extracted but no category was found in the expected "
                            f"train/good + test/<subtype> layout - inspect the extracted files under {root}.")
    return root

def download_mpdd():
    root = os.path.join(BASE_DATASETS_DIR, DATASETS["mpdd"]["root_subdir"])
    os.makedirs(BASE_DATASETS_DIR, exist_ok=True)
    if find_category_dirs_generic(root):
        print("  MPDD already present.")
        return root
    print("  MPDD not found locally. Downloading from Kaggle...")
    ensure_kaggle_credentials()
    os.system(f"kaggle datasets download -d {DATASETS['mpdd']['kaggle_slug']} -p {root} --unzip")
    if not find_category_dirs_generic(root):
        raise RuntimeError(f"MPDD download failed or has an unexpected folder layout - "
                            f"inspect the extracted files under {root}.")
    return root

def download_btad():
    root = os.path.join(BASE_DATASETS_DIR, DATASETS["btad"]["root_subdir"])
    os.makedirs(BASE_DATASETS_DIR, exist_ok=True)
    if find_category_dirs_generic(root):
        print("  BTAD already present.")
        return root
    print("  BTAD not found locally. Downloading from Kaggle...")
    ensure_kaggle_credentials()
    os.system(f"kaggle datasets download -d {DATASETS['btad']['kaggle_slug']} -p {root} --unzip")
    if not find_category_dirs_generic(root):
        raise RuntimeError(
            f"BTAD download failed, or its folders don't match 'train/<ok>' + 'test/<ok|ko>' - "
            f"inspect the extracted files under {root} (BTAD commonly uses 'ok'/'ko', which "
            f"GOOD_NAMES already covers, but re-check nesting/capitalization).")
    return root


def collect_dataset_tasks(dataset_key):

    if dataset_key == "mvtec_ad":
        root = download_mvtec()
        cat_dirs = find_category_dirs_generic(root)
    elif dataset_key == "mvtec_loco_ad":
        root = download_mvtec_loco()
        cat_dirs = find_category_dirs_generic(root)
    elif dataset_key == "mpdd":
        root = download_mpdd()
        cat_dirs = find_category_dirs_generic(root)
    elif dataset_key == "btad":
        root = download_btad()
        cat_dirs = find_category_dirs_generic(root)
    else:
        raise ValueError(f"Unknown dataset key: {dataset_key}")

    tasks = []
    for cat_dir in cat_dirs:
        name = os.path.basename(cat_dir)
        train_p, test_p, test_y, test_dt = get_mvtec_style_paths(cat_dir)
        if train_p and test_p:
            tasks.append({"category": name, "train_p": train_p, "test_p": test_p,
                           "test_y": test_y, "test_defect_types": test_dt})
    return tasks





print("Loading DINOv2 (ViT-L/14)...")
dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(DEVICE).eval()
if USE_FP16:
    dinov2_model = dinov2_model.half()
print("DINOv2 loaded.")

DINO_NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
RESIZE_CROP = T.Compose([T.Resize(RESIZE_SIZE), T.CenterCrop(IMG_SIZE)])

class ImageDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img = Image.open(self.file_paths[idx]).convert("RGB")
        img = RESIZE_CROP(img)
        t = DINO_NORM(TF.to_tensor(img))
        return {"dino": t}

def pil_to_model_tensor(pil_img):
    t = DINO_NORM(TF.to_tensor(pil_img)).unsqueeze(0).to(DEVICE)
    if USE_FP16:
        t = t.half()
    return t

def extract_cls_embeddings(paths, desc="cls", batch_size=32):
    if len(paths) == 0:
        return torch.zeros((0, 1))
    loader = DataLoader(ImageDataset(paths), batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)
    embs = []
    with torch.no_grad():
        for b in tqdm(loader, desc=desc, leave=False):
            dino_in = b["dino"].to(DEVICE)
            if USE_FP16:
                dino_in = dino_in.half()
            feats = dinov2_model.get_intermediate_layers(
                dino_in, n=1, reshape=False, return_class_token=True
            )
            cls_tok = feats[0][1].float()
            embs.append(cls_tok.cpu())
    return torch.cat(embs, dim=0)

@torch.no_grad()
def extract_cls_from_tensor(img_tensor):
    feats = dinov2_model.get_intermediate_layers(img_tensor, n=1, reshape=False, return_class_token=True)
    return feats[0][1].float().cpu()



def run_supervised(train_p, test_p, test_y):
    all_paths = train_p + test_p
    all_labels = np.array([0] * len(train_p) + list(test_y))
    embeddings = extract_cls_embeddings(all_paths, desc="supervised-embed").numpy()

    n_pos = all_labels.sum()
    n_folds = min(SUPERVISED_CV_FOLDS, max(2, n_pos)) if n_pos > 1 else 2
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    probs = np.zeros(len(all_labels))
    fold_id = np.full(len(all_labels), -1)
    fold_aurocs = []
    for f, (tr_idx, te_idx) in enumerate(skf.split(embeddings, all_labels)):
        clf = LogisticRegression(max_iter=3000, class_weight='balanced')
        clf.fit(embeddings[tr_idx], all_labels[tr_idx])
        probs[te_idx] = clf.predict_proba(embeddings[te_idx])[:, 1]
        fold_id[te_idx] = f
        if len(np.unique(all_labels[te_idx])) > 1:
            fold_aurocs.append(roc_auc_score(all_labels[te_idx], probs[te_idx]) * 100)

    overall_auroc = roc_auc_score(all_labels, probs) * 100

    explain_clf = LogisticRegression(max_iter=3000, class_weight='balanced')
    explain_clf.fit(embeddings, all_labels)

    return {
        "overall_auroc": overall_auroc,
        "fold_aurocs": fold_aurocs,
        "probs": probs,
        "labels": all_labels,
        "fold_id": fold_id,
        "paths": all_paths,
        "embeddings": embeddings,
        "explain_clf": explain_clf,
    }



def _attn_forward_capture(self, x, attn_bias=None):
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    attn = (q @ k.transpose(-2, -1)) * self.scale
    attn = attn.softmax(dim=-1)
    self._captured_attn = attn.detach()
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    x = self.proj(x)
    return x

def compute_attention_rollout(model, img_tensor, discard_ratio=DISCARD_RATIO_ROLLOUT):
    original_forwards = {}
    for i, blk in enumerate(model.blocks):
        original_forwards[i] = blk.attn.forward
        blk.attn.forward = types.MethodType(_attn_forward_capture, blk.attn)
    try:
        with torch.no_grad():
            _ = model(img_tensor)
        attn_mats = [blk.attn._captured_attn.mean(dim=1)[0].float() for blk in model.blocks]
    finally:
        for i, blk in enumerate(model.blocks):
            blk.attn.forward = original_forwards[i]

    result = torch.eye(attn_mats[0].size(0), device=attn_mats[0].device)
    for A in attn_mats:
        if discard_ratio > 0:
            flat = A.flatten()
            k = int(flat.numel() * discard_ratio)
            if k > 0:
                thresh = flat.kthvalue(k).values
                A = torch.where(A < thresh, torch.zeros_like(A), A)
        A = A + torch.eye(A.size(0), device=A.device)
        A = A / A.sum(dim=-1, keepdim=True)
        result = A @ result

    cls_to_patches = result[0, 1:]
    grid = int(round(cls_to_patches.numel() ** 0.5))
    if grid * grid != cls_to_patches.numel():
        raise ValueError(f"Non-square patch grid ({cls_to_patches.numel()} tokens) - "
                          f"check for register tokens / different image size.")
    heatmap = cls_to_patches.reshape(grid, grid).cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap

def safe_attention_rollout(model, pil_img):
    try:
        t = pil_to_model_tensor(pil_img)
        return compute_attention_rollout(model, t)
    except Exception as e:
        print(f"  [attention rollout skipped] {e}")
        return None



def compute_shap_values(explain_clf, embeddings, background_size=100, sample_size=200, seed=0):
    rng = np.random.RandomState(seed)
    bg_idx = rng.choice(len(embeddings), size=min(background_size, len(embeddings)), replace=False)
    sample_idx = rng.choice(len(embeddings), size=min(sample_size, len(embeddings)), replace=False)
    background = embeddings[bg_idx]
    sample = embeddings[sample_idx]

    try:
        explainer = shap.LinearExplainer(explain_clf, background)
        shap_values = explainer.shap_values(sample)
    except TypeError:
        masker = shap.maskers.Independent(background)
        explainer = shap.LinearExplainer(explain_clf, masker)
        sv = explainer(sample)
        shap_values = sv.values

    return shap_values, sample, sample_idx



def occlusion_saliency_map(pil_img_cropped, clf, grid=OCCLUSION_GRID):
    arr = np.array(pil_img_cropped)
    H, W = arr.shape[0], arr.shape[1]
    ch, cw = H // grid, W // grid
    mean_val = arr.reshape(-1, arr.shape[-1]).mean(axis=0).astype(np.uint8)

    base_emb = extract_cls_from_tensor(pil_to_model_tensor(pil_img_cropped)).numpy()
    base_prob = clf.predict_proba(base_emb)[0, 1]

    heat = np.zeros((grid, grid))
    for i in range(grid):
        for j in range(grid):
            occ = arr.copy()
            occ[i*ch:(i+1)*ch, j*cw:(j+1)*cw] = mean_val
            emb = extract_cls_from_tensor(pil_to_model_tensor(Image.fromarray(occ))).numpy()
            p = clf.predict_proba(emb)[0, 1]
            heat[i, j] = base_prob - p
    return heat, base_prob, mean_val

def insertion_deletion_curves(pil_img_cropped, clf, heat, mean_val, grid=OCCLUSION_GRID, n_steps=None):
    arr = np.array(pil_img_cropped)
    H, W = arr.shape[0], arr.shape[1]
    ch, cw = H // grid, W // grid
    order_saliency = np.dstack(np.unravel_index(np.argsort(-heat.ravel()), heat.shape))[0]
    n_steps = n_steps or (grid * grid)

    rng = np.random.RandomState(0)
    order_random = order_saliency.copy()
    rng.shuffle(order_random)

    def run_curve(order, mode):
        base = arr.copy() if mode == "deletion" else np.tile(mean_val, (H, W, 1)).astype(np.uint8)
        scores = []
        for step in range(n_steps + 1):
            emb = extract_cls_from_tensor(pil_to_model_tensor(Image.fromarray(base))).numpy()
            scores.append(clf.predict_proba(emb)[0, 1])
            if step < n_steps:
                i, j = order[step]
                if mode == "deletion":
                    base[i*ch:(i+1)*ch, j*cw:(j+1)*cw] = mean_val
                else:
                    base[i*ch:(i+1)*ch, j*cw:(j+1)*cw] = arr[i*ch:(i+1)*ch, j*cw:(j+1)*cw]
        return np.array(scores)

    del_saliency = run_curve(order_saliency, "deletion")
    ins_saliency = run_curve(order_saliency, "insertion")
    del_random = run_curve(order_random, "deletion")
    ins_random = run_curve(order_random, "insertion")

    x = np.linspace(0, 1, n_steps + 1)
    return {
        "deletion_auc_saliency": np.trapz(del_saliency, x),
        "insertion_auc_saliency": np.trapz(ins_saliency, x),
        "deletion_auc_random": np.trapz(del_random, x),
        "insertion_auc_random": np.trapz(ins_random, x),
        "curves": {"del_sal": del_saliency, "ins_sal": ins_saliency,
                   "del_rand": del_random, "ins_rand": ins_random, "x": x},
    }



def save_table(df, path_noext, index=False):
    df.to_csv(f"{path_noext}.csv", index=index)
    with open(f"{path_noext}.md", "w") as f:
        f.write(df.to_markdown(index=index))
    print(f"  Saved {os.path.basename(path_noext)}: {df.shape[0]} rows")
    return df

def generate_tables(dataset_key, display_name, tasks, category_results, xai_records, shap_records, tables_dir):
    if not category_results:
        print(f"  No results for {display_name} - skipping tables.")
        return None

    rows = []
    for task in tasks:
        n_train, n_test = len(task["train_p"]), len(task["test_p"])
        n_defect = int(sum(task["test_y"]))
        n_good_test = n_test - n_defect
        subtypes = sorted(set(dt for dt, y in zip(task["test_defect_types"], task["test_y"]) if y == 1))
        rows.append({"Category": task["category"], "Train (good)": n_train, "Test (good)": n_good_test,
                      "Test (defect)": n_defect, "Total": n_train + n_test,
                      "# Defect subtypes": len(subtypes), "Defect subtypes": ", ".join(subtypes)})
    save_table(pd.DataFrame(rows), f"{tables_dir}/table1_dataset_overview")

    rows = []
    for cat, res in category_results.items():
        rows.append({"Category": cat, "AUROC (overall, OOF)": round(res["overall_auroc"], 2),
                      "AUROC (fold mean)": round(float(np.mean(res["fold_aurocs"])), 2),
                      "AUROC (fold std)": round(float(np.std(res["fold_aurocs"])), 2),
                      "N images": len(res["labels"])})
    df2 = pd.DataFrame(rows)
    mean_row = {"Category": "Mean (all categories)",
                "AUROC (overall, OOF)": round(df2["AUROC (overall, OOF)"].mean(), 2),
                "AUROC (fold mean)": round(df2["AUROC (fold mean)"].mean(), 2),
                "AUROC (fold std)": round(df2["AUROC (fold std)"].mean(), 2),
                "N images": int(df2["N images"].sum())}
    table2 = save_table(pd.concat([df2, pd.DataFrame([mean_row])], ignore_index=True),
                         f"{tables_dir}/table2_main_results")

    rows = []
    for cat, res in category_results.items():
        y, p = res["labels"], res["probs"]
        fpr, tpr, thr = roc_curve(y, p)
        j = tpr - fpr
        best_thr = thr[np.argmax(j)]
        pred = (p >= best_thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        acc = (tp + tn) / (tp + tn + fp + fn)
        prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        rec = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else np.nan
        auprc = average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan
        mcc = matthews_corrcoef(y, pred) if len(np.unique(y)) > 1 else np.nan
        rows.append({"Category": cat, "Threshold*": round(float(best_thr), 3),
                      "AUROC": round(res["overall_auroc"], 2),
                      "AUPRC": round(float(auprc), 3) if not np.isnan(auprc) else auprc,
                      "Accuracy": round(acc, 3), "Precision": round(prec, 3), "Recall (Sens.)": round(rec, 3),
                      "Specificity": round(spec, 3), "F1": round(f1, 3),
                      "MCC": round(float(mcc), 3) if not np.isnan(mcc) else mcc,
                      "TP": tp, "FP": fp, "TN": tn, "FN": fn})
    table3 = save_table(pd.DataFrame(rows), f"{tables_dir}/table3_confusion_metrics")

    rows = []
    rng = np.random.RandomState(0)
    for cat, res in category_results.items():
        y, p = res["labels"], res["probs"]
        boots = []
        n = len(y)
        for _ in range(BOOTSTRAP_ITERS):
            idx = rng.randint(0, n, n)
            if len(np.unique(y[idx])) < 2:
                continue
            boots.append(roc_auc_score(y[idx], p[idx]) * 100)
        if boots:
            lo, hi = np.percentile(boots, [2.5, 97.5])
        else:
            lo, hi = np.nan, np.nan
        rows.append({"Category": cat, "AUROC": round(res["overall_auroc"], 2),
                      "95% CI low": round(lo, 2) if not np.isnan(lo) else lo,
                      "95% CI high": round(hi, 2) if not np.isnan(hi) else hi})
    table4 = save_table(pd.DataFrame(rows), f"{tables_dir}/table4_bootstrap_ci")

    rows = []
    for cat, res in category_results.items():
        row = {"Category": cat}
        for f, auroc in enumerate(res["fold_aurocs"]):
            row[f"Fold {f+1}"] = round(auroc, 2)
        rows.append(row)
    table5 = save_table(pd.DataFrame(rows), f"{tables_dir}/table5_fold_breakdown")

    lit_rows = [
        {"Method": "SPADE (Cohen & Hoshen, 2020)", "Setting": "Unsupervised (memory bank, no fine-tuning)", "Reported image AUROC (%)": "~85.5 (MVTec AD)"},
        {"Method": "PaDiM (Defard et al., 2021)", "Setting": "Unsupervised (Gaussian modeling)", "Reported image AUROC (%)": "~95.5-97.9 (MVTec AD)"},
        {"Method": "CFlow-AD (Gudovskiy et al., 2022)", "Setting": "Unsupervised (normalizing flow)", "Reported image AUROC (%)": "~98.3 (MVTec AD)"},
        {"Method": "PatchCore (Roth et al., 2022)", "Setting": "Unsupervised (coreset memory bank)", "Reported image AUROC (%)": "~99.1 (MVTec AD)"},
        {"Method": "WinCLIP (Jeong et al., 2023)", "Setting": "Zero-shot (CLIP + text prompts)", "Reported image AUROC (%)": "~91.8 (MVTec AD)"},
        {"Method": f"This work - Scenario A ({display_name})", "Setting": "Supervised upper bound (DINOv2 CLS + logistic regression, 5-fold CV)",
         "Reported image AUROC (%)": f"{np.mean([r['overall_auroc'] for r in category_results.values()]):.2f} (ours, measured)"},
    ]
    table6 = save_table(pd.DataFrame(lit_rows), f"{tables_dir}/table6_literature_comparison")

    table7 = None
    if xai_records:
        rows = []
        for rec in xai_records:
            f = rec["faithfulness"]
            rows.append({"Category": rec["category"], "Image": os.path.basename(rec["path"]),
                          "Deletion AUC (saliency)": round(f["deletion_auc_saliency"], 3),
                          "Deletion AUC (random)": round(f["deletion_auc_random"], 3),
                          "Insertion AUC (saliency)": round(f["insertion_auc_saliency"], 3),
                          "Insertion AUC (random)": round(f["insertion_auc_random"], 3)})
        df7 = pd.DataFrame(rows)
        agg7 = df7.drop(columns=["Category", "Image"]).mean().to_frame().T
        agg7.insert(0, "Category", "Mean (all sampled images)")
        table7 = save_table(pd.concat([df7, agg7], ignore_index=True), f"{tables_dir}/table7_xai_faithfulness")

    table8 = None
    if shap_records:
        all_shap = np.concatenate([v["shap_values"] for v in shap_records.values()], axis=0)
        mean_abs_shap = np.abs(all_shap).mean(axis=0)
        top_k = 15
        top_idx = np.argsort(-mean_abs_shap)[:top_k]
        table8 = save_table(pd.DataFrame({"Embedding dim": top_idx,
                                           "Mean |SHAP|": np.round(mean_abs_shap[top_idx], 4)}),
                             f"{tables_dir}/table8_shap_top_dims")

    n_params_dino = sum(p.numel() for p in dinov2_model.parameters())
    rows = [
        {"Stage": "DINOv2 ViT-L/14 backbone", "Parameters (M)": round(n_params_dino / 1e6, 1),
         "Notes": "Frozen, feature extraction only"},
        {"Stage": "Logistic regression probe (per fold)", "Parameters (M)": round(1024 / 1e6, 4),
         "Notes": "1024-D embedding -> 1 logit"},
        {"Stage": "Occlusion saliency (per image)", "Parameters (M)": np.nan,
         "Notes": f"{OCCLUSION_GRID**2} forward passes/image"},
        {"Stage": "Insertion/Deletion (per image)", "Parameters (M)": np.nan,
         "Notes": f"~{4*(OCCLUSION_GRID**2)} forward passes/image (both curves, both orderings)"},
    ]
    table9 = save_table(pd.DataFrame(rows), f"{tables_dir}/table9_computational_cost")

    rows = []
    for cat, res in category_results.items():
        y, p, dt = res["labels"], res["probs"], np.array(res["test_defect_types"])
        good_p = p[y == 0]
        for subtype in sorted(set(dt[y == 1])):
            sub_p = p[(y == 1) & (dt == subtype)]
            yy = np.array([0]*len(good_p) + [1]*len(sub_p))
            pp = np.concatenate([good_p, sub_p])
            if len(np.unique(yy)) < 2:
                continue
            rows.append({"Category": cat, "Defect subtype": subtype,
                          "N (defect)": len(sub_p), "AUROC vs. good": round(roc_auc_score(yy, pp) * 100, 2)})
    table10 = save_table(pd.DataFrame(rows), f"{tables_dir}/table10_defect_subtype_breakdown")

    master = table2[table2["Category"] != "Mean (all categories)"].merge(
        table3.drop(columns=["AUROC"]), on="Category", how="left")
    master = master.merge(table4.drop(columns=["AUROC"]), on="Category", how="left")
    master.insert(0, "Dataset", display_name)
    master_mean = master.drop(columns=["Category", "Dataset"]).mean(numeric_only=True).to_frame().T
    master_mean.insert(0, "Dataset", display_name)
    master_mean.insert(1, "Category", "Mean (all categories)")
    table_all_metrics = save_table(pd.concat([master, master_mean], ignore_index=True),
                                    f"{tables_dir}/table_ALL_METRICS_{dataset_key}")

    print(f"  All tables saved for {display_name}.")
    return {"table1": None, "table2": table2, "table3": table3, "table4": table4,
            "table5": table5, "table6": table6, "table7": table7, "table8": table8,
            "table9": table9, "table10": table10, "table_all_metrics": table_all_metrics}



def savefig(fig, path_noext):
    fig.savefig(f"{path_noext}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {os.path.basename(path_noext)}.png")

def generate_figures(dataset_key, display_name, category_results, xai_records, shap_records, figures_dir, xai_dir):
    if not category_results:
        print(f"  No results for {display_name} - skipping figures.")
        return

    rows = [{"Category": cat, "AUROC (overall, OOF)": res["overall_auroc"],
             "AUROC (fold std)": float(np.std(res["fold_aurocs"]))} for cat, res in category_results.items()]
    d = pd.DataFrame(rows).sort_values("AUROC (overall, OOF)")
    fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * len(d))))
    ax.barh(d["Category"], d["AUROC (overall, OOF)"], xerr=d["AUROC (fold std)"], color="#3B6EA5", capsize=3)
    ax.set_xlabel("AUROC (%)")
    ax.set_title(f"Scenario A (Supervised) - Per-Category AUROC on {display_name}")
    ax.set_xlim(min(80, d["AUROC (overall, OOF)"].min() - 5), 101)
    savefig(fig, f"{figures_dir}/fig1_per_category_auroc")

    fig, ax = plt.subplots(figsize=(7, 7))
    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    for cat, res in category_results.items():
        fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
        ax.plot(fpr, tpr, alpha=0.25, lw=1, color="gray")
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    macro_auroc = np.mean([r['overall_auroc'] for r in category_results.values()])
    ax.plot(mean_fpr, mean_tpr, color="#C0392B", lw=2.5, label=f"Macro-average (AUROC={macro_auroc:.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves - {display_name} (gray=per-category, red=macro-average)")
    ax.legend(loc="lower right")
    savefig(fig, f"{figures_dir}/fig2_roc_curves")

    fig, ax = plt.subplots(figsize=(7, 7))
    for cat, res in category_results.items():
        prec, rec, _ = precision_recall_curve(res["labels"], res["probs"])
        ax.plot(rec, prec, alpha=0.4, lw=1)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"Precision-Recall Curves - {display_name}")
    savefig(fig, f"{figures_dir}/fig3_pr_curves")

    agg_cm = np.zeros((2, 2), dtype=int)
    for cat, res in category_results.items():
        y, p = res["labels"], res["probs"]
        fpr, tpr, thr = roc_curve(y, p)
        best_thr = thr[np.argmax(tpr - fpr)]
        pred = (p >= best_thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        agg_cm += np.array([[tn, fp], [fn, tp]])
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(agg_cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Pred: Good", "Pred: Defect"], yticklabels=["True: Good", "True: Defect"], ax=ax)
    ax.set_title(f"Aggregate Confusion Matrix - {display_name}\n(Youden's J threshold, all categories)")
    savefig(fig, f"{figures_dir}/fig4_confusion_matrix")

    pooled_embs, pooled_labels = [], []
    for cat, res in category_results.items():
        pooled_embs.append(res["embeddings"]); pooled_labels.append(res["labels"])
    pooled_embs = np.concatenate(pooled_embs, axis=0)
    pooled_labels = np.concatenate(pooled_labels, axis=0)
    if len(pooled_embs) >= 10:
        rng = np.random.RandomState(0)
        sub_idx = rng.choice(len(pooled_embs), size=min(2000, len(pooled_embs)), replace=False)
        tsne = TSNE(n_components=2, random_state=0, init="pca",
                     perplexity=min(30, max(5, len(sub_idx)//4)))
        proj = tsne.fit_transform(pooled_embs[sub_idx])
        fig, ax = plt.subplots(figsize=(7, 6))
        for lab, name, color in [(0, "Good", "#2E86AB"), (1, "Defect", "#C0392B")]:
            mask = pooled_labels[sub_idx] == lab
            ax.scatter(proj[mask, 0], proj[mask, 1], s=8, alpha=0.6, label=name, color=color)
        ax.set_title(f"t-SNE of DINOv2 CLS Embeddings - {display_name} (pooled, subsampled)")
        ax.legend(); ax.set_xticks([]); ax.set_yticks([])
        savefig(fig, f"{figures_dir}/fig5_tsne_embeddings")

    if shap_records:
        rep_cat = list(shap_records.keys())[0]
        sv = shap_records[rep_cat]["shap_values"]
        shap.summary_plot(sv, shap_records[rep_cat]["sample"], show=False, plot_size=(8, 6))
        plt.title(f"SHAP Summary - {display_name} / {rep_cat} (logistic-regression probe)")
        savefig(plt.gcf(), f"{figures_dir}/fig6_shap_summary")

    if xai_records:
        n_show = min(6, len(xai_records))
        fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
        axes = np.atleast_2d(axes)
        for k in range(n_show):
            rec = xai_records[k]
            axes[0, k].imshow(rec["pil_img"]); axes[0, k].set_title(rec["category"], fontsize=9); axes[0, k].axis("off")
            hm = rec["occ_heat"]
            axes[1, k].imshow(rec["pil_img"])
            axes[1, k].imshow(np.kron(hm, np.ones((IMG_SIZE // hm.shape[0], IMG_SIZE // hm.shape[1]))),
                               cmap="jet", alpha=0.45)
            axes[1, k].axis("off")
        fig.suptitle(f"Occlusion Sensitivity - {display_name} Sample Defect Images")
        savefig(fig, f"{figures_dir}/fig7_occlusion_gallery")

    rollout_avail = [r for r in xai_records if r["rollout_map"] is not None]
    if rollout_avail:
        n_show = min(6, len(rollout_avail))
        fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
        axes = np.atleast_2d(axes)
        for k in range(n_show):
            rec = rollout_avail[k]
            axes[0, k].imshow(rec["pil_img"]); axes[0, k].set_title(rec["category"], fontsize=9); axes[0, k].axis("off")
            hm = rec["rollout_map"]
            up = np.kron(hm, np.ones((IMG_SIZE // hm.shape[0], IMG_SIZE // hm.shape[1])))
            axes[1, k].imshow(rec["pil_img"]); axes[1, k].imshow(up, cmap="jet", alpha=0.45); axes[1, k].axis("off")
        fig.suptitle(f"Attention Rollout - {display_name} Sample Defect Images")
        savefig(fig, f"{figures_dir}/fig8_attention_rollout_gallery")
    else:
        print(f"  No attention-rollout maps available for {display_name} - fig8 skipped.")

    all_p = np.concatenate([r["probs"] for r in category_results.values()])
    all_y = np.concatenate([r["labels"] for r in category_results.values()])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(all_p[all_y == 0], bins=30, alpha=0.6, label="Good", color="#2E86AB", density=True)
    axes[0].hist(all_p[all_y == 1], bins=30, alpha=0.6, label="Defect", color="#C0392B", density=True)
    axes[0].set_xlabel("Predicted P(defect)"); axes[0].set_ylabel("Density"); axes[0].legend()
    axes[0].set_title(f"Predicted Probability Distribution - {display_name}")
    bins = np.linspace(0, 1, 11)
    bin_idx = np.clip(np.digitize(all_p, bins) - 1, 0, 9)
    obs_freq = [all_y[bin_idx == b].mean() if np.any(bin_idx == b) else np.nan for b in range(10)]
    axes[1].plot(bins[:-1] + 0.05, obs_freq, "o-", color="#3B6EA5")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1].set_xlabel("Mean predicted probability (bin)"); axes[1].set_ylabel("Observed defect frequency")
    axes[1].set_title("Calibration Curve")
    savefig(fig, f"{figures_dir}/fig9_probability_calibration")

    if xai_records:
        del_sal = np.mean([r["faithfulness"]["curves"]["del_sal"] for r in xai_records], axis=0)
        ins_sal = np.mean([r["faithfulness"]["curves"]["ins_sal"] for r in xai_records], axis=0)
        del_rand = np.mean([r["faithfulness"]["curves"]["del_rand"] for r in xai_records], axis=0)
        ins_rand = np.mean([r["faithfulness"]["curves"]["ins_rand"] for r in xai_records], axis=0)
        x = xai_records[0]["faithfulness"]["curves"]["x"]
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].plot(x, del_sal, label="Saliency order", color="#C0392B")
        axes[0].plot(x, del_rand, label="Random order", color="gray", ls="--")
        axes[0].set_title("Deletion Curve (lower = more faithful)"); axes[0].set_xlabel("Fraction occluded")
        axes[0].set_ylabel("P(defect)"); axes[0].legend()
        axes[1].plot(x, ins_sal, label="Saliency order", color="#2E86AB")
        axes[1].plot(x, ins_rand, label="Random order", color="gray", ls="--")
        axes[1].set_title("Insertion Curve (higher = more faithful)"); axes[1].set_xlabel("Fraction revealed")
        axes[1].set_ylabel("P(defect)"); axes[1].legend()
        fig.suptitle(f"XAI Faithfulness - {display_name}")
        savefig(fig, f"{figures_dir}/fig10_insertion_deletion_curves")

   
    def _upsample_heatmap_smooth(hm, out_size=IMG_SIZE, sigma=1.2):
        zf = out_size / hm.shape[0]
        up = ndi_zoom(hm, zf, order=3)
        up = ndi_gaussian_filter(up, sigma=sigma)
        return up

    def _anomaly_localization(up, pct=75):
        thresh = float(np.percentile(up, pct))
        mask = up >= thresh
        labeled, n_comp = ndi_label(mask)
        if n_comp == 0:
            return None, thresh
        sizes = np.bincount(labeled.ravel())
        sizes[0] = 0
        largest = int(sizes.argmax())
        ys, xs = np.where(labeled == largest)
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))  # x0,y0,x1,y1
        return bbox, thresh

    def _draw_localization(ax, up, bbox, thresh):
        ax.contour(up, levels=[thresh], colors="white", linewidths=1.3, alpha=0.9)
        if bbox is not None:
            x0, y0, x1, y1 = bbox
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                    edgecolor="#39FF14", linewidth=2.0, linestyle="--"))

    if xai_records:
        seen_cats = set()
        gallery_picks = []
        for rec in xai_records:

            if rec["category"] not in seen_cats:
                gallery_picks.append(rec); seen_cats.add(rec["category"])
            if len(gallery_picks) == XAI_GALLERY_N_IMAGES:
                break

        i = 0
        while len(gallery_picks) < min(XAI_GALLERY_N_IMAGES, len(xai_records)):
            if xai_records[i] not in gallery_picks:
                gallery_picks.append(xai_records[i])
            i += 1

        n = len(gallery_picks)
        HEAT_CMAP = "turbo"
        fig, axes = plt.subplots(3, n, figsize=(4.4 * n, 13))
        if n == 1:
            axes = axes.reshape(3, 1)
        fig.suptitle(f"XAI Heatmap Gallery - {display_name}\n"
                     "Occlusion-Sensitivity Saliency with Anomaly Localization",
                     fontsize=15, fontweight="bold", y=0.995)
        last_im = None
        for k, rec in enumerate(gallery_picks):
            up = _upsample_heatmap_smooth(rec["occ_heat"])
            bbox, thresh = _anomaly_localization(up, pct=75)
            prob = rec.get("occ_base_prob", None)
            prob_str = f"P(defect) = {prob:.2f}" if prob is not None else ""

            axes[0, k].imshow(rec["pil_img"])
            axes[0, k].set_title(f"{rec['category']}\n{os.path.basename(rec['path'])}\n{prob_str}",
                                  fontsize=10)
            axes[0, k].axis("off")
            for spine in axes[0, k].spines.values():
                spine.set_visible(True); spine.set_edgecolor("#888888")

            axes[1, k].imshow(rec["pil_img"])
            last_im = axes[1, k].imshow(up, cmap=HEAT_CMAP, alpha=0.5, vmin=up.min(), vmax=up.max())
            _draw_localization(axes[1, k], up, bbox, thresh)
            axes[1, k].axis("off")
            axes[1, k].set_title("Localized anomaly region", fontsize=9.5, color="#333333")

            axes[2, k].imshow(up, cmap=HEAT_CMAP)
            _draw_localization(axes[2, k], up, bbox, thresh)
            axes[2, k].axis("off")

        axes[0, 0].text(-0.12, 0.5, "Original", transform=axes[0, 0].transAxes, rotation=90,
                         va="center", ha="center", fontsize=11, fontweight="bold")
        axes[1, 0].text(-0.12, 0.5, "Overlay", transform=axes[1, 0].transAxes, rotation=90,
                         va="center", ha="center", fontsize=11, fontweight="bold")
        axes[2, 0].text(-0.12, 0.5, "Heatmap", transform=axes[2, 0].transAxes, rotation=90,
                         va="center", ha="center", fontsize=11, fontweight="bold")

        fig.subplots_adjust(right=0.9, wspace=0.06, hspace=0.12)
        cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.55])
        fig.colorbar(last_im, cax=cbar_ax, label="Occlusion saliency (drop in P(defect))")
        legend_patch = mpatches.Patch(edgecolor="#39FF14", facecolor="none",
                                       label="Localized anomaly (top-25% saliency, largest region)")
        fig.legend(handles=[legend_patch], loc="lower center", bbox_to_anchor=(0.46, -0.01),
                   ncol=1, fontsize=9.5, frameon=False)
        gallery_path = f"{figures_dir}/fig11_xai_heatmap_gallery_3anomalies"
        fig.savefig(f"{gallery_path}.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved fig11_xai_heatmap_gallery_3anomalies.png ({n} anomaly image(s))")

        for k, rec in enumerate(gallery_picks):
            up = _upsample_heatmap_smooth(rec["occ_heat"])
            bbox, thresh = _anomaly_localization(up, pct=75)
            prob = rec.get("occ_base_prob", None)
            fig, ax = plt.subplots(figsize=(4.2, 4.2))
            ax.imshow(rec["pil_img"])
            ax.imshow(up, cmap="turbo", alpha=0.5, vmin=up.min(), vmax=up.max())
            _draw_localization(ax, up, bbox, thresh)
            ax.axis("off")
            title = f"{rec['category']}"
            if prob is not None:
                title += f" (P(defect)={prob:.2f})"
            ax.set_title(title, fontsize=10)
            fig.savefig(f"{xai_dir}/anomaly_{k+1}_{rec['category']}_heatmap.png", dpi=300, bbox_inches="tight")
            plt.close(fig)
    else:
        print(f"  No XAI records available for {display_name} - XAI heatmap gallery skipped.")

    print(f"  All figures saved for {display_name}.")


def create_pipeline_flowchart():
    stages = [
        ("1. Data Collection", "4 datasets: MVTec AD, MVTec LOCO AD,\nMPDD, BTAD", "#2E4057"),
        ("2. Preprocessing", "Resize (384) -> Center-crop (336)\nImageNet normalization", "#3B6EA5"),
        ("3. Feature Extraction", "DINOv2 ViT-L/14 (frozen)\nCLS token, 1024-D embedding", "#4C8C4A"),
        ("4. Supervised Classification\n(Scenario A)", "5-fold stratified CV\nClass-balanced logistic regression", "#C08A2E"),
        ("5. Explainable AI (XAI)", "Attention Rollout | SHAP (Linear)\nOcclusion + Insertion/Deletion", "#A8493A"),
        ("6. Evaluation & Reporting", "AUROC/AUPRC/MCC tables,\nfigures, XAI galleries per dataset", "#6A4C93"),
    ]

    fig, ax = plt.subplots(figsize=(6.5, 15))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(stages) * 3 + 1)
    ax.axis("off")

    box_w, box_h = 8.6, 2.1
    x0 = (10 - box_w) / 2
    ys = []
    for idx, (title, subtitle, color) in enumerate(stages):
        y_top = len(stages) * 3 - idx * 3
        ys.append(y_top)
        box = FancyBboxPatch((x0, y_top - box_h), box_w, box_h,
                              boxstyle="round,pad=0.12,rounding_size=0.18",
                              linewidth=1.4, edgecolor="#222222", facecolor=color, alpha=0.92)
        ax.add_patch(box)
        ax.text(5, y_top - box_h * 0.35, title, ha="center", va="center",
                fontsize=12.5, fontweight="bold", color="white")
        ax.text(5, y_top - box_h * 0.72, subtitle, ha="center", va="center",
                fontsize=9.5, color="white", linespacing=1.4)

    for idx in range(len(stages) - 1):
        y_start = ys[idx] - box_h
        y_end = ys[idx + 1]
        arrow = FancyArrowPatch((5, y_start), (5, y_end + 0.05),
                                 arrowstyle="-|>", mutation_scale=22,
                                 linewidth=1.6, color="#333333")
        ax.add_patch(arrow)

    ax.set_title("Explainable Supervised Anomaly Classification Pipeline\n"
                  "(DINOv2 + Logistic Regression, Multi-Dataset)",
                  fontsize=13.5, fontweight="bold", pad=18)

    out_path = f"{ASSETS_DIR}/figures/fig0_pipeline_flowchart"
    fig.savefig(f"{out_path}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved fig0_pipeline_flowchart.png")



def run_dataset_pipeline(dataset_key):
    cfg = DATASETS[dataset_key]
    display_name = cfg["display_name"]
    out_dir = os.path.join(ASSETS_DIR, dataset_key)
    tables_dir = os.path.join(out_dir, "tables")
    figures_dir = os.path.join(out_dir, "figures")
    xai_dir = os.path.join(out_dir, "xai_examples")
    for d in (tables_dir, figures_dir, xai_dir):
        os.makedirs(d, exist_ok=True)

    print("\n" + "=" * 78)
    print(f"DATASET: {display_name}")
    print("=" * 78)

    print(f"Collecting {display_name} categories...")
    tasks = collect_dataset_tasks(dataset_key)
    print(f"Found {len(tasks)} categories/products.")
    if not tasks:
        print(f"  WARNING: no categories found for {display_name} - skipping this dataset entirely. "
              f"Check the folder layout under {BASE_DATASETS_DIR}/{cfg['root_subdir']}.")
        return None
    if DEBUG_MODE:
        tasks = tasks[:DEBUG_N_CATEGORIES]
        print(f"DEBUG_MODE: running only {len(tasks)} categories.")

    category_results, xai_records, shap_records = {}, [], {}

    t_start = time.time()
    for i, task in enumerate(tasks):
        cat = task["category"]
        print(f"\n[{i+1}/{len(tasks)}] {display_name} / {cat}")
        res = run_supervised(task["train_p"], task["test_p"], task["test_y"])
        res["test_defect_types"] = ["good"] * len(task["train_p"]) + task["test_defect_types"]
        category_results[cat] = res
        print(f"  AUROC = {res['overall_auroc']:.2f} "
              f"(fold mean {np.mean(res['fold_aurocs']):.2f} +/- {np.std(res['fold_aurocs']):.2f})")

        shap_values, shap_sample, shap_idx = compute_shap_values(res["explain_clf"], res["embeddings"])
        shap_records[cat] = {"shap_values": shap_values, "sample": shap_sample}

        defect_paths = [p for p, y in zip(task["test_p"], task["test_y"]) if y == 1]
        rng = np.random.RandomState(0)
        chosen = (rng.choice(defect_paths, size=min(XAI_SAMPLES_PER_CATEGORY, len(defect_paths)), replace=False)
                  if defect_paths else [])

        for img_path in chosen:
            pil_img = RESIZE_CROP(Image.open(img_path).convert("RGB"))
            rollout_map = safe_attention_rollout(dinov2_model, pil_img)
            occ_heat, occ_base_prob, occ_mean_val = occlusion_saliency_map(pil_img, res["explain_clf"])
            faith = insertion_deletion_curves(pil_img, res["explain_clf"], occ_heat, occ_mean_val)
            xai_records.append({
                "category": cat, "path": img_path, "pil_img": pil_img,
                "rollout_map": rollout_map, "occ_heat": occ_heat, "occ_base_prob": occ_base_prob,
                "faithfulness": faith,
            })
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n{display_name} runtime: {(time.time() - t_start)/60:.1f} min")
    print(f"Mean AUROC across {len(category_results)} categories: "
          f"{np.mean([r['overall_auroc'] for r in category_results.values()]):.2f}")

    print(f"\nGenerating tables for {display_name}...")
    tables = generate_tables(dataset_key, display_name, tasks, category_results, xai_records, shap_records, tables_dir)
    print(f"\nGenerating figures for {display_name}...")
    generate_figures(dataset_key, display_name, category_results, xai_records, shap_records, figures_dir, xai_dir)

    return {"tasks": tasks, "category_results": category_results,
            "xai_records": xai_records, "shap_records": shap_records, "tables": tables}


create_pipeline_flowchart()

all_results = {}
t_global_start = time.time()
for dataset_key in DATASETS.keys():
    try:
        all_results[dataset_key] = run_dataset_pipeline(dataset_key)
    except Exception as e:
        print(f"\n!!! {DATASETS[dataset_key]['display_name']} FAILED: {e}")
        print("    Continuing with the remaining datasets.\n")
        all_results[dataset_key] = None

print(f"\nTotal multi-dataset runtime: {(time.time() - t_global_start)/60:.1f} min")



summary_rows = []
for dataset_key, res in all_results.items():
    display_name = DATASETS[dataset_key]["display_name"]
    if res is None or not res["category_results"]:
        summary_rows.append({"Dataset": display_name, "Status": "FAILED / NO DATA",
                              "# Categories": 0, "Mean AUROC": np.nan, "Total N images": 0})
        continue
    aurocs = [r["overall_auroc"] for r in res["category_results"].values()]
    n_images = sum(len(r["labels"]) for r in res["category_results"].values())
    summary_rows.append({"Dataset": display_name, "Status": "OK",
                          "# Categories": len(res["category_results"]),
                          "Mean AUROC": round(float(np.mean(aurocs)), 2),
                          "Total N images": n_images})
cross_dataset_summary = pd.DataFrame(summary_rows)
cross_dataset_summary.to_csv(f"{ASSETS_DIR}/cross_dataset_summary.csv", index=False)
with open(f"{ASSETS_DIR}/cross_dataset_summary.md", "w") as f:
    f.write(cross_dataset_summary.to_markdown(index=False))
print("\n" + "=" * 78)
print("CROSS-DATASET SUMMARY")
print("=" * 78)
print(cross_dataset_summary.to_string(index=False))



_DATASET_COLORS = ["#C0392B", "#2E86AB", "#4C8C4A", "#C08A2E", "#6A4C93",
                    "#A8493A", "#3B6EA5", "#8E44AD"]

roc_overlay_curves = {}
for i, (dataset_key, res) in enumerate(all_results.items()):
    if res is None or not res["category_results"]:
        continue
    display_name = DATASETS[dataset_key]["display_name"]
    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    for cat, r in res["category_results"].items():
        fpr, tpr, _ = roc_curve(r["labels"], r["probs"])
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
    macro_auroc = float(np.mean([r["overall_auroc"] for r in res["category_results"].values()]))
    roc_overlay_curves[dataset_key] = {"fpr": mean_fpr, "tpr": mean_tpr, "auroc": macro_auroc,
                                        "display_name": display_name}

if roc_overlay_curves:
    fig, ax = plt.subplots(figsize=(8, 8))
    for i, (dataset_key, c) in enumerate(sorted(roc_overlay_curves.items(),
                                                  key=lambda kv: -kv[1]["auroc"])):
        color = _DATASET_COLORS[i % len(_DATASET_COLORS)]
        ax.plot(c["fpr"], c["tpr"], lw=2.2, color=color,
                label=f"{c['display_name']} (AUROC = {c['auroc']:.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("Macro-Average ROC Curves - All Datasets\n"
                  "(Scenario A: DINOv2 CLS + Logistic Regression)", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right", fontsize=9.5, framealpha=0.92)
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
    ax.grid(alpha=0.3)
    fig.savefig(f"{ASSETS_DIR}/figures/fig_cross_dataset_roc_overlay.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved fig_cross_dataset_roc_overlay.png (Task 2: all datasets' ROC curves in one figure)")
else:
    print("No dataset produced results - skipping cross-dataset ROC overlay figure.")

ok_summary = cross_dataset_summary[cross_dataset_summary["Status"] == "OK"]
if len(ok_summary) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(ok_summary["Dataset"], ok_summary["Mean AUROC"], color="#3B6EA5")
    ax.set_ylabel("Mean AUROC (%)")
    ax.set_title("Scenario A Mean AUROC by Dataset")
    ax.set_ylim(min(80, ok_summary["Mean AUROC"].min() - 5), 101)
    plt.xticks(rotation=20, ha="right")
    fig.savefig(f"{ASSETS_DIR}/figures/fig_cross_dataset_auroc_summary.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved fig_cross_dataset_auroc_summary.png")



all_metrics_frames = []
for dataset_key, res in all_results.items():
    if res is None or not res.get("tables") or res["tables"].get("table_all_metrics") is None:
        continue
    all_metrics_frames.append(res["tables"]["table_all_metrics"])

if all_metrics_frames:
    table_all_datasets = pd.concat(all_metrics_frames, ignore_index=True)

    headline_cols = ["Dataset", "Category", "AUROC (overall, OOF)", "Accuracy",
                      "Precision", "Recall (Sens.)", "F1"]
    other_cols = [c for c in table_all_datasets.columns if c not in headline_cols]
    table_all_datasets = table_all_datasets[[c for c in headline_cols if c in table_all_datasets.columns] + other_cols]
    save_table(table_all_datasets, f"{ASSETS_DIR}/table_ALL_DATASETS_ALL_METRICS")
    print(f"Saved table_ALL_DATASETS_ALL_METRICS: {table_all_datasets.shape[0]} rows across "
          f"{len(all_metrics_frames)} dataset(s) (Task 4: Accuracy/Precision/Recall/F1/AUROC + extras).")
else:
    print("No per-dataset metrics tables available - skipping cross-dataset all-metrics table.")



import shutil
shutil.make_archive("paper_assets", "zip", ASSETS_DIR)
print("\nPackaged: paper_assets.zip")

try:
    from google.colab import files
    files.download("paper_assets.zip")
except ImportError:
    print("Not running in Colab - paper_assets.zip is in the working directory, "
          "download it manually from the Files sidebar.")

Device: cuda
Loading DINOv2 (ViT-L/14)...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:08<00:00, 143MB/s]


DINOv2 loaded.
Saved fig0_pipeline_flowchart.png

DATASET: MVTec AD
  MVTec AD not found locally. Downloading from Kaggle...
Upload your kaggle.json:


Saving kaggle.json to kaggle.json
Found 15 categories/products.

[1/15] MVTec AD / bottle


  AUROC = 100.00 (fold mean 100.00 +/- 0.00)

[2/15] MVTec AD / cable


...

  AUROC = 98.61 (fold mean 98.83 +/- 1.65)

[2/3] BTAD (BeanTech Anomaly Detection) / 02


  AUROC = 93.10 (fold mean 93.38 +/- 2.16)

[3/3] BTAD (BeanTech Anomaly Detection) / 03


  AUROC = 100.00 (fold mean 100.00 +/- 0.00)

BTAD (BeanTech Anomaly Detection) runtime: 2.7 min
Mean AUROC across 3 categories: 97.23

Generating tables for BTAD (BeanTech Anomaly Detection)...
  Saved table1_dataset_overview: 3 rows
  Saved table2_main_results: 4 rows
  Saved table3_confusion_metrics: 3 rows
  Saved table4_bootstrap_ci: 3 rows
  Saved table5_fold_breakdown: 3 rows
  Saved table6_literature_comparison: 6 rows
  Saved table7_xai_faithfulness: 10 rows
  Saved table8_shap_top_dims: 15 rows
  Saved table9_computational_cost: 4 rows
  Saved table10_defect_subtype_breakdown: 3 rows
  Saved table_ALL_METRICS_btad: 4 rows
  All tables saved for BTAD (BeanTech Anomaly Detection).

Generating figures for BTAD (BeanTech Anomaly Detection)...
  Saved fig1_per_category_auroc.png
  Saved fig2_roc_curves.png
  Saved fig3_pr_curves.png
  Saved fig4_confusion_matrix.png
  Saved fig5_tsne_embeddings.png
  Saved fig6_shap_summary.png
  Saved fig7_occlusion_gallery.png
  Saved fig8_atten

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>